In [ ]:
# Import the tools used in this notebook:
# - pandas/numpy for data tables and math
# - matplotlib for the prediction plot
# - sklearn for the train/test split, model, and scores
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [ ]:
# Load the Boston housing training data.
# Each row is one house area, and MEDV is the median house price in $1000s.
df = pd.read_csv("train.csv")
df.head()


In [ ]:
# Check every column for missing (null) values.
# If the total is 0, the table is already complete.
missing = df.isna().sum()
print(missing)
print("Total missing values:", int(missing.sum()))


In [ ]:
# Clean the data before modeling.
# ID is only a row label, so it cannot explain price and is removed.
# dropna() removes any rows that still have empty cells.
df = df.drop(columns="ID")
df = df.dropna()

# X = input features (what we know about the house)
# y = target value (the price we want to predict)
X = df.drop(columns="MEDV").copy()
y = df["MEDV"]

# Extra features help the model catch curved and combined price patterns:
# RM2 / LSTAT2 = non-linear effects of rooms and lower-status share
# RM_LSTAT = rooms and lower-status share together
# log_CRIM = compresses extreme crime rates
# TAX_PTRATIO = tax burden relative to school crowding
X["RM2"] = X["RM"] ** 2
X["LSTAT2"] = X["LSTAT"] ** 2
X["RM_LSTAT"] = X["RM"] * X["LSTAT"]
X["log_CRIM"] = np.log1p(X["CRIM"])
X["TAX_PTRATIO"] = X["TAX"] / X["PTRATIO"]

print("Features (X):", list(X.columns))
print("Target (y): MEDV")
print("Rows after cleaning:", len(df))


In [ ]:
# Hold out 20% of the houses so we can test the model on data it did not train on.
# random_state=42 keeps the split the same every time we run the notebook.
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Gradient boosting builds many small trees.
# Each new tree tries to correct the mistakes of the previous trees.
# These settings were tuned for higher R2 and lower MSE:
# - more trees + smaller learning_rate = careful learning
# - subsample / max_features = less overfitting
model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.04,
    max_depth=3,
    subsample=0.8,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
)

# Train on the training houses, then predict prices for the validation houses.
model.fit(X_train, y_train)
y_hat = model.predict(X_valid)

# Score how close the predicted prices are to the real prices.
# R2: share of price differences explained (higher is better)
# MSE: average squared miss (lower is better)
# Average error: average absolute miss in $1000s (lower is better)
r2 = r2_score(y_valid, y_hat) * 100
mse = mean_squared_error(y_valid, y_hat)
average_error = mean_absolute_error(y_valid, y_hat)

print("Gradient boosting")
print("R2:", round(r2, 1), "%")
print("MSE:", round(mse, 2))
print("Average error:", round(average_error, 2))

# Scatter plot: each point is one house.
# The red line is a perfect prediction (actual == predicted).
plt.figure(figsize=(6, 5))
plt.scatter(y_valid, y_hat, alpha=0.7)
low = min(y_valid.min(), y_hat.min())
high = max(y_valid.max(), y_hat.max())
plt.plot([low, high], [low, high], color="crimson", linewidth=1)
plt.xlabel("Actual price, y ($1000s)")
plt.ylabel("Predicted price, y_hat ($1000s)")
plt.title(f"Gradient boosting\nR2: {r2:.1f}% \nMSE: {mse:.2f}")
plt.tight_layout()
plt.show()
